# 🧪 Lab 5: The Legacy Partner Terminal (Hierarchical Parsing & Contract Validation Foundation)

In this laboratory session, we open the legacy partner terminal to evaluate the built-in XML capabilities introduced in Apache Spark 4. Instead of relying on external third-party plug-ins or complex string manipulation routines, we will leverage Spark 4's native XML datasource and inline column functions. We will implement structural path configurations, process tag-level attributes, evaluate multi-element array widening scenarios, and establish explicit boundaries to isolate syntax-level corruptions from downstream business logic violations.

### Step 1: Initialize Local Spark Workspace
We spin up an isolated standalone Spark 4 session and check our baseline engine parameters to confirm that hierarchical formats are supported natively.

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import os

spark = SparkSession.builder.getOrCreate()
base_scratch_dir = "/tmp/spark_flight_xml_lab5"
os.makedirs(base_scratch_dir, exist_ok=True)

print(f"Active Spark Engine Version: {spark.version}")
print(f"Staging Workspace Location: {base_scratch_dir}")

Active Spark Engine Version: 4.1.2
Staging Workspace Location: /tmp/spark_flight_xml_lab5


### Step 2: Fabricate Hierarchical Legacy Payloads
We generate an alternate transactional dataset mimicking multi-supplier partner data where key business variables like loyalty tiers, carriers, and baggage counts are explicitly stored as element attributes rather than flat text values.

In [8]:
xml_records = [
    ("ROW-01", "SUP-A", '<Booking id="BKG-001" carrier="IB"><supplier_id>SUP-A</supplier_id><passenger loyalty_tier="GOLD">John Doe</passenger><route><segment seq="1" from="MAD" to="BCN"/></route><fare family="EconomyLight" code="ECO-LGT"><price currency="EUR">87.50</price></fare><baggage quantity="1" unit="PC"/><timestamp>2026-06-16T12:00:00Z</timestamp></Booking>'),
    ("ROW-02", "SUP-B", '<Booking id="BKG-002" carrier="AF"><supplier_id>SUP-B</supplier_id><passenger loyalty_tier="NONE">Alice Smith</passenger><passenger loyalty_tier="SILVER">Bob Smith</passenger><route><segment seq="1" from="CDG" to="AMS"/><segment seq="2" from="AMS" to="JFK"/></route><fare family="Business" code="BUS-CLS"><price currency="USD">2450.00</price></fare><baggage quantity="2" unit="PC"/><timestamp>2026-06-16T13:00:00Z</timestamp></Booking>'),
    ("ROW-03", "SUP-C", '<Booking id="BKG-BROKEN" carrier="LH"><supplier_id>SUP-C</supplier_id><passenger loyalty_tier="NONE">Truncated Tag Missing Element</Booking>'),
    ("ROW-04", "SUP-D", '<Booking id="BKG-NEGPRICE" carrier="UA"><supplier_id>SUP-D</supplier_id><passenger loyalty_tier="PLATINUM">Charlie Brown</passenger><route><segment seq="1" from="EWR" to="ORD"/></route><fare family="EconomyClassic" code="ECO-CLS"><price currency="USD">-150.00</price></fare><baggage quantity="0" unit="PC"/><timestamp>2026-06-16T14:00:00Z</timestamp></Booking>')
]

# Save raw text fields into an ingestion DataFrame container
raw_stream_df = spark.createDataFrame(xml_records, ["row_id", "supplier_id", "raw_payload"])

# Output contents to a scratch directory to validate the file-based DataFrame reader
xml_file_path = f"{base_scratch_dir}/raw_bookings.xml"
with open(xml_file_path, "w") as f:
    f.write("<Bookings>\n" + "\n".join([r[2] for r in xml_records]) + "\n</Bookings>")

print(f"Hierarchical testing streams successfully cached at: {xml_file_path}")

Hierarchical testing streams successfully cached at: /tmp/spark_flight_xml_lab5/raw_bookings.xml


### Step 3: Track 1 — Native File-Based XML Ingestion
We read the staged XML file using Spark 4's built-in file reader, applying a target row tag config to establish clear relational record slices.

In [9]:
# Load structural paths natively from disk using rowTag alignment
file_ingested_df = spark.read.format("xml") \
    .option("rowTag", "Booking") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .load(xml_file_path)

print("=== TRACK 1: NATIVE FILE INGESTION LAYER VIEW ===")
file_ingested_df.show(truncate=False)
file_ingested_df.printSchema()

=== TRACK 1: NATIVE FILE INGESTION LAYER VIEW ===
+--------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Step 4: Schema Inference Blueprint Evaluation
We execute schema inference on a minimized row sample via `schema_of_xml` to trace how automated structural discovery overlooks downstream multi-passenger group collections.

In [10]:
economy_sample = xml_records[0][2]

# Extract inferred DDL string structure from economy string container using range(1) workaround
inferred_ddl = spark.range(1).select(F.schema_of_xml(F.lit(economy_sample))).collect()[0][0]

print("=== STEP 4: AUTOMATED INFERENCE DDL LIGHT ===")
print(inferred_ddl)

=== STEP 4: AUTOMATED INFERENCE DDL LIGHT ===
STRUCT<_carrier: STRING, _id: STRING, baggage: STRUCT<_quantity: BIGINT, _unit: STRING>, fare: STRUCT<_code: STRING, _family: STRING, price: STRUCT<_VALUE: DOUBLE, _currency: STRING>>, passenger: STRUCT<_VALUE: STRING, _loyalty_tier: STRING>, route: STRUCT<segment: STRUCT<_from: STRING, _seq: BIGINT, _to: STRING>>, supplier_id: STRING, timestamp: TIMESTAMP>


### Step 5: Track 2 — Column Function Processing via Explicit Contracts
We parse inline raw text columns using `from_xml` coupled with a robust, hand-declared explicit schema to absorb attribute prefixes and array-widening shifts safely.

In [11]:
# Hand-crafted production schema designed to absorb multiple passengers, segmented routes, and custom attribute tags
explicit_ddl_contract = (
    "_id STRING, _carrier STRING, supplier_id STRING, "
    "passenger ARRAY<STRUCT<_loyalty_tier: STRING, _VALUE: STRING>>, "
    "route STRUCT<segment: ARRAY<STRUCT<_from: STRING, _seq: INT, _to: STRING>>>, "
    "fare STRUCT<_code: STRING, _family: STRING, price: STRUCT<_currency: STRING, _VALUE: DECIMAL(10,2)>>, "
    "baggage STRUCT<_quantity: INT, _unit: STRING>, timestamp STRING, _corrupt_record STRING"
)

# Map column elements inline via explicit DDL rules
parsed_inline_df = raw_stream_df.withColumn(
    "parsed_struct",
    F.from_xml(F.col("raw_payload"), explicit_ddl_contract, {"mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt_record"})
).cache()

print("=== TRACK 2: EXPLICIT SCHEMATIC PARSING OUTCOMES ===")
parsed_inline_df.select("row_id", "parsed_struct.*").show(truncate=False)

=== TRACK 2: EXPLICIT SCHEMATIC PARSING OUTCOMES ===
+------+------------+--------+-----------+------------------------------------------+--------------------------------+-----------------------------------------+-------+--------------------+--------------------------------------------------------------------------------------------------------------------------------------------+
|row_id|_id         |_carrier|supplier_id|passenger                                 |route                           |fare                                     |baggage|timestamp           |_corrupt_record                                                                                                                             |
+------+------------+--------+-----------+------------------------------------------+--------------------------------+-----------------------------------------+-------+--------------------+------------------------------------------------------------------------------------------------

### Step 6: Defensive Boundary Auditing & Quality Controls
We execute downstream business verification logic beside our parser configurations to demonstrate the security boundary dividing structural syntax rules from data quality audits.

In [12]:
production_serving_df = parsed_inline_df.select(
    F.coalesce(F.col("parsed_struct._id"), F.lit("UNKNOWN")).alias("booking_id"),
    F.col("parsed_struct._carrier").alias("carrier"),
    F.col("parsed_struct.fare.price._VALUE").alias("total_price"),
    F.col("parsed_struct.passenger._loyalty_tier").alias("loyalty_tiers"),
    F.col("parsed_struct._corrupt_record").alias("syntax_corrupt_payload"),
    F.when(F.col("parsed_struct._corrupt_record").isNotNull(), F.lit("CORRUPT_SYNTAX"))
     .when(F.col("parsed_struct.fare.price._VALUE") < 0, F.lit("INVALID_BUSINESS_DATA"))
     .otherwise(F.lit("VALID")).alias("gate_security_status")
)

print("=== STEP 6: PRODUCTION QUALITY RADAR REPORT ===")
production_serving_df.show(truncate=False)

# Compute terminal metrics report summary
metrics_summary = production_serving_df.groupBy("gate_security_status").count()
print("=== TERMINAL INGESTION SUMMARY COUNTS ===")
metrics_summary.show(truncate=False)

=== STEP 6: PRODUCTION QUALITY RADAR REPORT ===
+------------+-------+-----------+--------------+--------------------------------------------------------------------------------------------------------------------------------------------+---------------------+
|booking_id  |carrier|total_price|loyalty_tiers |syntax_corrupt_payload                                                                                                                      |gate_security_status |
+------------+-------+-----------+--------------+--------------------------------------------------------------------------------------------------------------------------------------------+---------------------+
|BKG-001     |IB     |87.50      |[GOLD]        |NULL                                                                                                                                        |VALID                |
|BKG-002     |AF     |2450.00    |[NONE, SILVER]|NULL                                               

## 📊 Post-Lab Analysis: Evidence from the Engine Room

### 1. File-Level Collapse vs. Inline Structural Isolation
* **The Macro Parser Panic:** Track 1 (Step 3) exposed the extreme dangers of file-level ingestion boundaries when dealing with raw syntax damage. When the file reader hit the truncated tag in `BKG-BROKEN`, the macro parser panicked and aborted normal processing loop boundaries—engulfing the entire multi-row XML payload into a single corrupt record string block and completely losing the subsequent, perfectly valid `BKG-NEGPRICE` transaction from the dataframe view.
* **Row-Isolated Lineage Preservation:** Conversely, Track 2 (Step 5) proved the absolute resilience of parsing raw text inline via column expressions. The `from_xml` function surgically quarantined the syntax failure of `ROW-03` inside an isolated row boundary, completely shielding the engine from a macro cluster collapse and allowing the downstream data pipeline to successfully ingest and evaluate `ROW-04`.

### 2. The Schema Inference Flashlight Gap
* **Structural Shape Miscalculation:** Step 4's automated inference check via `schema_of_xml` explicitly illustrated why dynamic discovery cannot substitute for strict production contracts. Because the automated engine room built its type rules solely from the single-passenger `ROW-01` baseline, it inferred both `passenger` and `route.segment` as flat, singular `STRUCT` nodes.
* **The Widening Tax:** Had this inferred blueprint been deployed directly to production, the pipeline would have immediately crashed or experienced massive data corruption the moment it encountered the multi-passenger groups and multi-leg connection matrices transmitted in `ROW-02`.

### 3. Compilation Taxes & The Double-Computation Penalty
* **Bytecode Generation Overhead:** The execution clocks revealed a heavy infrastructure cost associated with dynamic hierarchical string parsing. While loading the raw text records into memory was instantaneous (~0.99 seconds), compiling the inline `from_xml` expression against a complex DDL structural contract in Step 5 forced a substantial 18.32-second execution cycle as the Catalyst optimizer generated custom JVM evaluation bytecode.
* **The Lazy Evaluation Tax:** Step 6 incurred the heaviest latency penalty of the entire lab, consuming 27.03 seconds to clear execution threads. Because the underlying parsed DataFrame was not explicitly cached, calling consecutive terminal actions (`.show()` and `.groupBy().count()`) forced Spark to re-parse the raw XML payloads and re-generate the structural bytecode maps completely from scratch for each separate report.